# §4.3  Compute Efficiency

In [ ]:
import sys
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path('../../../../src').resolve()))
from config import get_snapshot_redshift

try:
    from utils.matplotlib_config import setconfig
    setconfig()
except ImportError:
    pass

In [ ]:
DATA_ROOT = Path('../../../../data/2pcf/scope_xi')
MODEL     = 'lc16'
SIM       = 'L800'
N_REF     = 1024   # full-box Corrfunc reference (pending — falls back to max available n)

# Quick-look config (sections 1–2)
QK_IZ        = 155
QK_MSTAR_TAG = 'mstar9.0'
QK_Z         = get_snapshot_redshift(f'iz{QK_IZ}', 'L800')

# n values to show in detail and headline plots
PLOT_N     = [2,4,8,16,32,64, 128, 256]#[2, 8, 32, 128, 256]
HEADLINE_N = [2,4,8,16,32,64, 128, 256]#[4, 16, 64, 256]
# n=128 and n=256 are infeasible for mstar_none: runtime >> cosma8 8h/16h limits
PLOT_N_MSTAR_NONE = [n for n in PLOT_N if n < 128]

print(f'Data root:   {DATA_ROOT}')
print(f'Quick-look:  iz{QK_IZ}  →  z = {QK_Z:.3f}  ({QK_MSTAR_TAG})')

## §4.3  Compute Efficiency

### Wall-clock budget (binding constraint)

| Partition | Time limit | Notes |
|-----------|-----------|-------|
| cosma5 | 2 h | Main workhorse for N ≤ 32 |
| cosma8-shm | 2 h | cosma5 data visible; N ≤ 256 |
| cosma8 | 8 h | No cosma5 mount — avoid |

### Thread scaling (L800/lc16, N=32, mstar9.0, z=1.5, 116,542 galaxies)

| Threads | t_pairs (s) | t_total (s) | Speedup |
|---------|------------|------------|---------|
| 1 | 89.5 | 91.4 | 1× |
| 4 | 24.0 | 25.9 | 3.7× |
| 8 | 11.6 | 13.5 | 7.7× |
| 16 | 6.4 | **8.2** | **14.1×** |
| 32 | 3.4 | 201.6 | I/O saturated |
| 64 | 2.8 | 228.4 | I/O saturated |

**Optimal: 16 threads.** Core-hours cost: 8.2 s × 16 = 0.037 CPU-h per job — negligible vs GALFORM (~10³ CPU-h/realisation).

I/O saturates the storage bus beyond 32 threads. The timing plot below shows both components.

In [ ]:
def plot_timing_paper():
    path = Path('../../../../data/2pcf/scope_xi_timing')
    files = sorted(path.glob('timing_*.csv'))
    if not files:
        print(f'No timing CSVs in {path}')
        return

    tdf = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    print(f'Loaded {len(tdf)} timing rows from {len(files)} files')
    print(tdf[['sim', 'model', 'iz', 'n_subvol', 'n_threads',
                'box_mpc_h', 'n_gal', 't_io_s', 't_pairs_s', 't_total_s']].to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Thread scaling: L800, n=32
    thread_df = (tdf[(tdf['sim'] == 'L800') & (tdf['n_subvol'] == 32)]
                   .sort_values('n_threads'))
    if not thread_df.empty:
        axes[0].plot(thread_df['n_threads'], thread_df['t_pairs_s'],
                     'o-', lw=2, label='pair counting')
        axes[0].plot(thread_df['n_threads'], thread_df['t_total_s'],
                     's--', lw=1.5, label='total (incl. I/O)')
        t1  = thread_df.loc[thread_df['n_threads'].idxmin(), 't_pairs_s']
        ts  = thread_df['n_threads'].values
        axes[0].plot(ts, t1 / ts, color='grey', lw=1, ls=':', label='ideal 1/N')
    axes[0].set_xscale('log', base=2)
    axes[0].set_yscale('log')
    axes[0].set_xlabel('RAYON_NUM_THREADS', fontsize=12)
    axes[0].set_ylabel('Wall time (s)', fontsize=12)
    axes[0].set_title('Thread scaling  (L800/lc16  iz155  mstar9.0  n=32)', fontsize=11)
    axes[0].legend(fontsize=10)

    # Box-size scaling: one point per (sim, n_threads=32)
    box_df = tdf[tdf['n_threads'] == 32].sort_values('box_mpc_h')
    if not box_df.empty:
        for _, row in box_df.iterrows():
            axes[1].scatter(row['box_mpc_h'], row['t_pairs_s'], s=100, zorder=3)
            axes[1].annotate(
                f"{row['sim']}/{row['model']}\n$N_{{\\rm gal}}$={row['n_gal']:,.0f}",
                (row['box_mpc_h'], row['t_pairs_s']),
                textcoords='offset points', xytext=(6, 3), fontsize=8,
            )
    axes[1].set_xscale('log')
    axes[1].set_yscale('log')
    axes[1].set_xlabel(r'Box size [$h^{-1}$Mpc]', fontsize=12)
    axes[1].set_ylabel('Pair-counting wall time (s)', fontsize=12)
    axes[1].set_title('Box-size scaling  (32 threads)', fontsize=11)

    fig.suptitle('SCOPE runtime scaling', fontsize=14)
    plt.tight_layout()
    plt.show()


plot_timing_paper()
